## Split learning محلی — **`CT_HTTPS`**

- داده: **`CHARTEVENTS.zip`** کنار کد (در ریپوی گیت‌هاب) → استخراج → **`CHARTEVENTS.csv`**
- جریان: `new_dataset` → `client_network` (ترنسفورمر + CLS) → شبیه‌سازی split → `prediction_net` (کپسولی)

سلول‌ها: **کلون ریپو** → **unzip** → **reload ماژول‌ها** → **هایپرپارامترها** → **آموزش**

In [ ]:
# اگر نوت‌بوک را از داخل همین پوشهٔ پروژه باز کرده‌ای (بدون clone) این را True بگذار
SKIP_CLONE = False

REPO_URL = "https://github.com/drfaranakyousefi-creator/fixed_project1_capsnet.git"
REPO_NAME = "fixed_project1_capsnet"

import os
import subprocess
import sys
from pathlib import Path

if not SKIP_CLONE:
    if not Path(REPO_NAME).is_dir():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(REPO_NAME)

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
print("cwd =", ROOT)

In [ ]:
import zipfile
from pathlib import Path

ZIP_PATH = Path("CHARTEVENTS.zip")
if not ZIP_PATH.is_file():
    raise FileNotFoundError(
        "فایل CHARTEVENTS.zip باید در ریشهٔ همین پوشه (کنار نوت‌بوک) باشد."
    )

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(".")

cands = sorted(Path(".").rglob("CHARTEVENTS.csv"))
if not cands:
    raise FileNotFoundError("بعد از unzip، CHARTEVENTS.csv پیدا نشد.")
CHARTEVENTS_CSV = str(cands[0].resolve())
print("CHARTEVENTS_CSV =", CHARTEVENTS_CSV)

In [ ]:
import importlib

import new_dataset
import client_net
import server_net
import transmitter_simulation
import Train_simulation

for _mod in (new_dataset, client_net, server_net, transmitter_simulation, Train_simulation):
    importlib.reload(_mod)

from Train_simulation import CT_HTTPS
import plot

print("ماژول‌ها reload شدند؛ CT_HTTPS آماده است.")

## تنظیمات (هایپرپارامترها)

مسیر CSV از سلول unzip می‌آید (`CHARTEVENTS_CSV`). اگر سلول unzip را اجرا نکردی، در سلول بعد می‌توانی مسیر را دستی بگذاری.

In [ ]:
from pathlib import Path

if "CHARTEVENTS_CSV" not in globals():
    CHARTEVENTS_CSV = str((Path.cwd() / "CHARTEVENTS.csv").resolve())

# ───────────────────────────────────────────
# پارامترهای داده
# ───────────────────────────────────────────
w              = 3
dataset_name   = "metavision"  # 'metavision' یا 'carevue'
batch_size     = 128
target         = "spO2"  # 'spO2' یا 'BP' یا 'RR'
test_size      = 0.2
normalize_data = True

# ───────────────────────────────────────────
# کلاینت — ترنسفورمر
# ───────────────────────────────────────────
client_lr       = 0.01
d_model         = 64
nhead           = 4
dim_feedforward = 128
num_cells       = 3
dropout         = 0.1

# ───────────────────────────────────────────
# سرور — شبکهٔ کپسولی روی CLS
# ───────────────────────────────────────────
server_lr         = 0.01
num_primary_caps  = 8
primary_dim       = 8
num_output_caps   = 4
output_dim        = 16
num_routing       = 3
server_fc_hidden1 = 128
server_fc_hidden2 = 64

# ───────────────────────────────────────────
# آموزش
# ───────────────────────────────────────────
epochs = 5
SEED   = 42

print("پارامترها OK")
print("  CSV:", CHARTEVENTS_CSV)
print("  w=%s dataset=%s batch=%s" % (w, dataset_name, batch_size))

In [ ]:
import random
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

trainer = CT_HTTPS(
    chartevents_path   = CHARTEVENTS_CSV,
    w                  = w,
    dataset_name       = dataset_name,
    batch_size         = batch_size,
    target             = target,
    test_size          = test_size,
    normalize_data     = normalize_data,
    client_lr          = client_lr,
    d_model            = d_model,
    nhead              = nhead,
    dim_feedforward    = dim_feedforward,
    num_cells          = num_cells,
    dropout            = dropout,
    server_lr          = server_lr,
    num_primary_caps   = num_primary_caps,
    primary_dim        = primary_dim,
    num_output_caps    = num_output_caps,
    output_dim         = output_dim,
    num_routing        = num_routing,
    server_fc_hidden1  = server_fc_hidden1,
    server_fc_hidden2  = server_fc_hidden2,
)

history = trainer.fit(epochs)
plot.plot_history(history)